# kdd_seq_02_next_token_gru: Causal Next-Token Prediction GRU

Trains an autoregressive unidirectional 2-layer GRU language model (`CausalGRULM`) on learner interaction sequences (`<bos> + tokens + <eos>`), extracts length-normalized causal hidden representations, clusters them in spherical latent space across $K=2\dots8$, and evaluates stability, behavioural profiles, and downstream dropout lift.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

WORK = "/content/drive/MyDrive/kdd_seq"
sys.path.insert(0, WORK)
import kdd_seq_common as C

# ------------------------------- CONFIG -------------------------------
METHOD = "gru_lm"
MAX_LEN = 128          # most recent 128 tokens (covers >95% of sequences completely)
BATCH_SIZE = 512
EPOCHS = 15
LR = 3e-3
WEIGHT_DECAY = 1e-4
K_GRID = list(range(2, 9))
K_FINAL = 8            # matches summary_markov.json; K=5 also evaluated downstream
DOWNSTREAM_KS = [5, 8] # compare both behavioural regime (K=5) and Markov BIC match (K=8)
MIN_SHARE = 0.03
CV_SAMPLE, CV_SPLITS, CV_REPEATS = 60000, 3, 3
SEEDS_STAB, N_BOOT = 5, 8
SEED = 42
# ----------------------------------------------------------------------
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(f"{WORK}/results", exist_ok=True)

A = C.load_artifacts(WORK)
V = A["vocab"]["V"]; NAMES = A["vocab"]["names"]
F_tr, F_te, y_tr, y_te = A["F_train"], A["F_test"], A["y_train"], A["y_test"]
nev_tr, nev_te = A["base_train"]["n_events"].values, A["base_test"]["n_events"].values
print(f"Device: {DEVICE} | Train: {F_tr.shape[0]} | Test: {F_te.shape[0]} | V={V}")


In [ ]:
# ==============================================================================
# Shared Sequence Packing, Spherical Clustering & Evaluation Helpers
# ==============================================================================
BOS = V          # 19
EOS = V + 1      # 20
PAD = V + 2      # 21
VOCAB_SIZE = V + 3  # 22

def pack_sequences(tokens, offsets, max_len=128):
    """Packs flat CSR sequences (tokens, offsets) into a dense 2D array (N, L_max+2)
    with BOS at index 0, up to max_len most recent tokens (preserving recency), EOS, and PAD."""
    N = len(offsets) - 1
    lengths = np.diff(offsets)
    L_cap = int(min(max_len, lengths.max()))
    out = np.full((N, L_cap + 2), PAD, dtype=np.int64)
    out[:, 0] = BOS
    eff_lens = np.minimum(lengths, L_cap).astype(np.int64)
    for i in range(N):
        s, e = offsets[i], offsets[i + 1]
        L = e - s
        if L > L_cap:
            s = e - L_cap
            L = L_cap
        out[i, 1:1 + L] = tokens[s:e]
        out[i, 1 + L] = EOS
    return out, eff_lens, lengths.astype(np.int64)


class SphericalClusterer:
    """K-Means on L2-normalized embeddings with descending cluster size ordering
    (cluster 0 = largest) and temperature-scaled cosine softmax posteriors."""
    def __init__(self, K, seed=0, n_init=10, temp=0.15):
        self.K = K
        self.seed = seed
        self.n_init = n_init
        self.temp = temp

    def fit(self, Z):
        km = KMeans(self.K, n_init=self.n_init, random_state=self.seed).fit(Z)
        counts = np.bincount(km.labels_, minlength=self.K)
        order = np.argsort(-counts)
        self.centroids_ = km.cluster_centers_[order]
        norms = np.linalg.norm(self.centroids_, axis=1, keepdims=True)
        self.centroids_norm_ = self.centroids_ / np.maximum(norms, 1e-12)
        self.inertia_ = float(km.inertia_)
        return self

    def cosine_sim(self, Z):
        Zn = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)
        return (Zn @ self.centroids_norm_.T).astype(np.float32)

    def predict_proba(self, Z):
        logits = self.cosine_sim(Z) / self.temp
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_l = np.exp(logits)
        return (exp_l / exp_l.sum(axis=1, keepdims=True)).astype(np.float32)

    def predict(self, Z):
        return self.cosine_sim(Z).argmax(axis=1)


def residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te):
    """Regresses out log1p(n_events) and log1p(seq_len) from embeddings (fitted on train only),
    then re-normalizes to unit L2 length."""
    X_tr = np.column_stack([np.ones(len(nev_tr)), np.log1p(nev_tr), np.log1p(len_tr)])
    X_te = np.column_stack([np.ones(len(nev_te)), np.log1p(nev_te), np.log1p(len_te)])
    W, _, _, _ = np.linalg.lstsq(X_tr, Z_tr, rcond=None)
    R_tr = Z_tr - X_tr @ W
    R_te = Z_te - X_te @ W
    R_tr /= np.maximum(np.linalg.norm(R_tr, axis=1, keepdims=True), 1e-12)
    R_te /= np.maximum(np.linalg.norm(R_te, axis=1, keepdims=True), 1e-12)
    return R_tr.astype(np.float32), R_te.astype(np.float32)


def describe_clusters_from_F(F, R_soft, V, names, alpha=0.1, cap=50, top=6, min_mass=0.005):
    """Describes any clustering using the empirical transition matrix F[:, V:] in the
    exact same characteristic lift & frequency format as the Markov mixture."""
    K = R_soft.shape[1]
    n = np.asarray(F[:, V:].sum(1)).ravel()
    Fc = (sp.diags(np.minimum(1.0, cap / np.maximum(n, 1))) @ F).tocsr() if cap else F
    S = np.asarray(Fc.T.dot(R_soft)).T[:, V:].reshape(K, V, V)
    G = S.sum(0) + alpha
    Pg = G / G.sum(1, keepdims=True)
    out = {}
    for k in range(K):
        Pk = (S[k] + alpha) / (S[k] + alpha).sum(1, keepdims=True)
        mass = S[k] / np.maximum(S[k].sum(), 1e-12)
        lift = np.where(mass >= min_mass, Pk / Pg, 0)
        li = np.dstack(np.unravel_index(np.argsort(-lift, axis=None)[:top], lift.shape))[0]
        fi = np.dstack(np.unravel_index(np.argsort(-mass, axis=None)[:top], mass.shape))[0]
        out[k] = dict(
            share=float(R_soft[:, k].mean()),
            characteristic=[f"{names[a]} -> {names[b]} (lift {lift[a,b]:.1f})" for a, b in li],
            frequent=[f"{names[a]} -> {names[b]} ({mass[a,b]*100:.1f}%)" for a, b in fi]
        )
    return out


X_tr, eff_tr, len_tr = pack_sequences(A["tok_train"], A["off_train"], MAX_LEN)
X_te, eff_te, len_te = pack_sequences(A["tok_test"],  A["off_test"],  MAX_LEN)
print("Packed train tensor:", X_tr.shape, "| Packed test tensor:", X_te.shape)


## 1. Neural Architecture & Unsupervised Pretraining
Trained strictly on `X_tr` without dropout labels (`y_tr`) to prevent any target leakage.

In [ ]:
class CausalGRULM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_emb=32, d_hid=64, n_layers=2, dropout=0.1, pad_idx=PAD):
        super().__init__()
        self.pad_idx = pad_idx
        self.emb = nn.Embedding(vocab_size, d_emb, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(d_emb, d_hid, num_layers=n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Linear(d_hid, vocab_size)

    def forward_hidden(self, x):
        h, _ = self.gru(self.drop(self.emb(x)))
        return h

    def loss(self, x):
        inp, tgt = x[:, :-1], x[:, 1:]
        logits = self.head(self.forward_hidden(inp))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=self.pad_idx)

    @torch.no_grad()
    def encode(self, x):
        inp = x[:, :-1]
        mask = (inp != self.pad_idx).unsqueeze(-1).float()
        h = self.forward_hidden(inp)
        mean_h = (h * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        lengths = mask.squeeze(-1).sum(1).long().clamp(min=1) - 1
        last_h = h[torch.arange(x.size(0), device=x.device), lengths]
        z = 0.5 * F.normalize(mean_h, p=2, dim=-1) + 0.5 * F.normalize(last_h, p=2, dim=-1)
        return F.normalize(z, p=2, dim=-1)

MODEL_CLS = CausalGRULM

idx_tr, idx_va = train_test_split(np.arange(len(X_tr)), test_size=0.1, random_state=SEED)
train_loader = DataLoader(TensorDataset(torch.tensor(X_tr[idx_tr], dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_tr[idx_va], dtype=torch.long)), batch_size=BATCH_SIZE * 2, shuffle=False)

model = MODEL_CLS().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best_val_loss, best_state = np.inf, None
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, n_tr = 0.0, 0
    for (bx,) in train_loader:
        bx = bx.to(DEVICE)
        opt.zero_grad()
        loss = model.loss(bx)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item() * len(bx); n_tr += len(bx)
    sched.step()
    model.eval()
    va_loss, n_va = 0.0, 0
    with torch.no_grad():
        for (bx,) in val_loader:
            bx = bx.to(DEVICE)
            va_loss += model.loss(bx).item() * len(bx); n_va += len(bx)
    tr_loss /= n_tr; va_loss /= n_va
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if ep == 1 or ep % 3 == 0 or ep == EPOCHS:
        print(f"Epoch {ep:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} (ppl={np.exp(va_loss):.2f}) | {time.time()-t0:.1f}s")

model.load_state_dict(best_state)
print(f"Best validation loss: {best_val_loss:.4f} (perplexity: {np.exp(best_val_loss):.2f})")


## 2. Embedding Extraction & Volume Residualization
Extracts L2-normalized embeddings (`Z_tr`, `Z_te`) and computes a volume-orthogonalized copy (`Z_tr_orth`, `Z_te_orth`) by regressing out $\log(1 + n_{\text{events}})$ and $\log(1 + L_{\text{seq}})$.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, X_np, batch_size=1024):
    model.eval()
    zs = []
    for i in range(0, len(X_np), batch_size):
        bx = torch.tensor(X_np[i:i + batch_size], dtype=torch.long, device=DEVICE)
        zs.append(model.encode(bx).cpu().numpy())
    return np.vstack(zs).astype(np.float32)

Z_tr = extract_embeddings(model, X_tr)
Z_te = extract_embeddings(model, X_te)
Z_tr_orth, Z_te_orth = residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te)

pca16 = PCA(n_components=min(16, Z_tr.shape[1]), random_state=0).fit(Z_tr)
Z_tr_pca = pca16.transform(Z_tr).astype(np.float32)
Z_te_pca = pca16.transform(Z_te).astype(np.float32)

pca16_orth = PCA(n_components=min(16, Z_tr_orth.shape[1]), random_state=0).fit(Z_tr_orth)
Z_tr_orth_pca = pca16_orth.transform(Z_tr_orth).astype(np.float32)
Z_te_orth_pca = pca16_orth.transform(Z_te_orth).astype(np.float32)

print("Z_tr:", Z_tr.shape, "| PCA16 explained var: %.3f" % pca16.explained_variance_ratio_.sum())


## 3. Clustering Grid ($K = 2 \dots 8$): Shared Space, Native Space & Volume Check

In [ ]:
rows, models = [], {}
for K in K_GRID:
    t0 = time.time()
    m = SphericalClusterer(K, seed=0, n_init=10).fit(Z_tr)
    models[K] = m
    lab_tr, lab_te = m.predict(Z_tr), m.predict(Z_te)
    P_tr = m.predict_proba(Z_tr)
    r = dict(K=K, inertia=m.inertia_, mean_max_posterior=float(P_tr.max(1).mean()))
    # Shared evaluation space (comparable across all methods including Markov)
    r.update(C.cluster_quality(A["eval_train"], lab_tr))
    # Native embedding space
    nat = C.cluster_quality(Z_tr, lab_tr)
    r["native_silhouette"] = nat["silhouette"]
    r["native_davies_bouldin"] = nat["davies_bouldin"]
    r["native_calinski_harabasz"] = nat["calinski_harabasz"]
    # Volume confounding & dropout association
    r.update(C.volume_agreement(lab_tr, nev_tr))
    r["cramers_v_train"] = C.cramers_v(lab_tr, y_tr)
    r["cramers_v_test"]  = C.cramers_v(lab_te, y_te)
    rows.append(r)
    print(f"K={K} done in {time.time()-t0:.1f}s | shared Sil={r['silhouette']:.4f} | native Sil={r['native_silhouette']:.4f} | ARI_vol={r['ari_vs_volume']:.4f}")

grid = pd.DataFrame(rows).set_index("K")
grid.round(4).to_csv(f"{WORK}/results/{METHOD}_grid.csv")
display(grid.round(4))


## 4. Cluster Profiles & Behavioural Interpretation (Characteristic Transitions)

In [ ]:
for K_view in DOWNSTREAM_KS:
    print(f"\n=================== {METHOD.upper()} PROFILE (K={K_view}) ===================")
    mk = models[K_view]
    l_tr = mk.predict(Z_tr)
    print(C.cluster_profile(l_tr, y_tr, nev_tr))
    desc = describe_clusters_from_F(F_tr, mk.predict_proba(Z_tr), V, NAMES)
    json.dump(desc, open(f"{WORK}/results/{METHOD}_cluster_descriptions_K{K_view}.json", "w"), indent=1)
    for k, d in desc.items():
        print(f"\n--- Cluster {k} (soft share {d['share']:.3f}) ---")
        print("  characteristic:", "; ".join(d["characteristic"][:4]))
        print("  frequent:      ", "; ".join(d["frequent"][:4]))


## 5. Stability Analysis (Seed, Bootstrap & Train $\rightarrow$ Test Transfer)

In [ ]:
stab_by_k = {}
for K_eval in DOWNSTREAM_KS:
    mk = models[K_eval]
    lab_tr_k, lab_te_k = mk.predict(Z_tr), mk.predict(Z_te)
    seed_labels = [SphericalClusterer(K_eval, seed=100 + s, n_init=5).fit(Z_tr).predict(Z_tr) for s in range(SEEDS_STAB)]
    st = C.seed_stability(seed_labels)
    st.update(C.bootstrap_stability(
        fit=lambda idx, seed: SphericalClusterer(K_eval, seed=seed, n_init=5).fit(Z_tr[idx]),
        predict=lambda cm, rows: cm.predict(Z_tr[rows]),
        n_rows=Z_tr.shape[0], ref_labels=lab_tr_k, n_boot=N_BOOT))
    st.update(C.transfer_consistency(lab_tr_k, y_tr, lab_te_k, y_te, K_eval))
    stab_by_k[K_eval] = {k: round(v, 4) for k, v in st.items()}
    print(f"\n=== Stability ({METHOD}, K={K_eval}) ===")
    print(pd.Series(stab_by_k[K_eval]).to_string())


## 6. Downstream Dropout Prediction (Train CV & Real Test Set)
Evaluates `hard`, `soft`, `hard+soft`, `cos_sim` (analogous to Markov `avgll`), continuous neural embeddings (`emb_pca16`), volume-orthogonalized embeddings (`emb_vol_orth`), and `null_random`.

In [ ]:
def make_variants(model_k, K, Z_a, Z_pca_a, Z_orth_pca_a, seed=0):
    P = model_k.predict_proba(Z_a)
    lab = P.argmax(1)
    cs = model_k.cosine_sim(Z_a)
    rng = np.random.default_rng(seed)
    return {
        "hard": C.onehot(lab, K),
        "soft": P.astype("float32"),
        "hard+soft": np.hstack([C.onehot(lab, K), P]).astype("float32"),
        "cos_sim": cs.astype("float32"),
        "emb_pca16": Z_pca_a.astype("float32"),
        "emb_vol_orth": Z_orth_pca_a.astype("float32"),
        "null_random": C.onehot(rng.integers(0, K, Z_a.shape[0]), K)
    }

cv_all, test_all = [], []
for K in DOWNSTREAM_KS:
    mk = models[K]
    var_tr = make_variants(mk, K, Z_tr, Z_tr_pca, Z_tr_orth_pca, seed=0)
    var_te = make_variants(mk, K, Z_te, Z_te_pca, Z_te_orth_pca, seed=1)
    summ, wide = C.downstream_cv(A["base_train"], y_tr, var_tr, CV_SAMPLE, CV_SPLITS, CV_REPEATS)
    summ.insert(0, "K", K); summ.insert(0, "method", METHOD); cv_all.append(summ)
    print(f"\n=== {METHOD} Train CV, K={K} ==="); display(summ)
    ft = C.final_test_eval(A["base_train"], y_tr, A["base_test"], y_te, var_tr, var_te)
    ft.insert(0, "K", K); ft.insert(0, "method", METHOD); test_all.append(ft)
    print(f"=== {METHOD} Real TEST Set, K={K} ==="); display(ft)

pd.concat(cv_all).to_csv(f"{WORK}/results/{METHOD}_downstream_cv.csv")
pd.concat(test_all).to_csv(f"{WORK}/results/{METHOD}_downstream_test.csv")


## 7. Save Artifacts & Summary JSON

In [ ]:
np.save(f"{WORK}/results/feat_{METHOD}_train_emb.npy", Z_tr)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb.npy",  Z_te)
np.save(f"{WORK}/results/feat_{METHOD}_train_emb_orth.npy", Z_tr_orth)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb_orth.npy",  Z_te_orth)
torch.save(model.state_dict(), f"{WORK}/results/{METHOD}_weights.pt")

for K_save in DOWNSTREAM_KS:
    mk = models[K_save]
    for s, ZZ in (("train", Z_tr), ("test", Z_te)):
        P = mk.predict_proba(ZZ)
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_soft.npy", P.astype("float32"))
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_cossim.npy", mk.cosine_sim(ZZ).astype("float32"))
        pd.DataFrame(P, index=A[f"ids_{s}"], columns=[f"p{k}" for k in range(K_save)]).assign(cluster=P.argmax(1)) \
          .to_csv(f"{WORK}/results/{METHOD}_labels_K{K_save}_{s}.csv", index_label="enrollment_id")

summary = dict(method=METHOD, K=K_FINAL, max_len=MAX_LEN, val_loss=float(best_val_loss))
summary.update({k: float(v) for k, v in grid.loc[K_FINAL].items()})
summary.update(stab_by_k[K_FINAL])
summary["K5_summary"] = {
    **{k: float(v) for k, v in grid.loc[5].items()},
    **stab_by_k[5]
}
json.dump(summary, open(f"{WORK}/results/summary_{METHOD}.json", "w"), indent=1)
print(json.dumps(summary, indent=1))
